<a href="https://colab.research.google.com/github/abcdofbigdata/agents/blob/main/anthropic/001_Prompt_Evals_complete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install boto3


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.4/139.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 100.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 8.6 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata

os.environ["AWS_ACCESS_KEY_ID"] = userdata.get('aws_access_key')
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get('aws_secret_access_key')

In [3]:
import boto3
import json

In [4]:
client = boto3.client("bedrock-runtime", region_name="us-west-2")
# Use Haiku for faster evals
model_id = "us.anthropic.claude-3-5-haiku-20241022-v1:0"


def add_user_message(messages, text):
    user_message = {"role": "user", "content": [{"text": text}]}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": [{"text": text}]}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {
            "temperature": temperature,
            "stopSequences": stop_sequences,
        },
    }

    if system:
        params["system"] = [{"text": system}]

    response = client.converse(**params)

    return response["output"]["message"]["content"][0]["text"]

In [5]:
def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [6]:
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [7]:
dataset

[{'task': 'Create a JSON configuration for an AWS Lambda function that sets up environment variables for database connection',
  'format': 'json',
  'solution_criteria': 'Must include valid Lambda function configuration with environment variables for host, username, password, and database name'},
 {'task': 'Write a Python function to extract the AWS account ID from an IAM role ARN',
  'format': 'python',
  'solution_criteria': 'Function should parse the ARN string and return only the 12-digit AWS account number, handle different ARN formats'},
 {'task': 'Develop a regular expression to validate an AWS S3 bucket name according to AWS naming rules',
  'format': 'regex',
  'solution_criteria': 'Regex must enforce rules: 3-63 characters, lowercase letters, numbers, periods, and hyphens, cannot start/end with period or hyphen'}]

In [8]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""

    system_prompt = "You are an experienced AWS engineer with a hyper focus on addressing corner cases"

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"], system=system_prompt)
    return output

In [9]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)


In [10]:
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [11]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [12]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [13]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 8.5


In [14]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\n{\n    \"LambdaFunctionConfiguration\": {\n        \"Environment\": {\n            \"Variables\": {\n                \"DB_HOST\": \"database-cluster.rds.amazonaws.com\",\n                \"DB_PORT\": \"5432\",\n                \"DB_NAME\": \"production_database\",\n                \"DB_USERNAME\": \"${ssm:/my-secure-param/username}\",\n                \"DB_PASSWORD\": \"${ssm:/my-secure-param/password}\",\n                \"DB_POOL_SIZE\": \"10\",\n                \"DB_TIMEOUT\": \"30\",\n                \"DB_SSL_MODE\": \"require\",\n                \"CONNECTION_MAX_RETRIES\": \"3\",\n                \"CONNECTION_RETRY_DELAY_MS\": \"500\"\n            }\n        }\n    }\n}\n",
    "test_case": {
      "task": "Create a JSON configuration for an AWS Lambda function that sets up environment variables for database connection",
      "format": "json",
      "solution_criteria": "Must include valid Lambda function configuration with environment variables for host, u